# food-vision: train on Google Colab

Runs the **same scripts** as the local pipeline on a free Colab GPU. Nothing here duplicates project logic.

1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` (🔑 icon in the left sidebar) holding a GitHub token with read access to the repo.
3. Run all cells. Outputs go to `runs_colab/` and `reports_colab/`, so they never overwrite the recorded results in `runs/` and `reports/`.

Approximate cost: the Food-101 download (~5 GB) takes a few minutes; training time depends on the GPU Colab assigns.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Get the code

In [ ]:
import os
REPO = "samadarsh/food-vision"

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # public repo, or not running on Colab

if not os.path.exists("food-vision"):
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    !git clone -q {url} food-vision
    # Don't leave the token in .git/config.
    !git -C food-vision remote set-url origin https://github.com/{REPO}.git
%cd food-vision
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 2. Configure

Starts from `configs/default.yaml`. Set `USE_VIT_BASE = True` to fine-tune the larger ViT-B/16, which is practical on a CUDA GPU.

In [ ]:
import yaml

USE_VIT_BASE = False   # ViT-B/16 (86M params) instead of ViT-S/16 (22M)
EPOCHS = None          # e.g. 5 for a quick run; None keeps the config value

cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["data"]["num_workers"] = 2             # Colab VMs have 2 CPU cores
cfg["data"]["reports_dir"] = "reports_colab"
cfg["train"]["runs_dir"] = "runs_colab"
if USE_VIT_BASE:
    cfg["models"]["vit"]["arch"] = "vit_base_patch16_224.augreg2_in21k_ft_in1k"
    cfg["models"]["vit"]["lr"] = 5.0e-5
if EPOCHS:
    cfg["train"]["epochs"] = EPOCHS
yaml.safe_dump(cfg, open("configs/colab.yaml", "w"), sort_keys=False)
print("vit arch:", cfg["models"]["vit"]["arch"], "| epochs:", cfg["train"]["epochs"])

## 3. Data (download + leak-free splits + EDA)

In [ ]:
!python scripts/prepare_data.py --config configs/colab.yaml
!python scripts/eda.py --config configs/colab.yaml

## 4. Models

In [ ]:
!python scripts/zero_shot_clip.py --config configs/colab.yaml

In [ ]:
!python scripts/train.py --model cnn --config configs/colab.yaml

In [ ]:
!python scripts/train.py --model vit --config configs/colab.yaml

## 5. Evaluate and explain

In [ ]:
!python scripts/evaluate.py --config configs/colab.yaml
!python scripts/explain.py --config configs/colab.yaml

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("reports_colab/eval/comparison.md").read()))
display(Image("reports_colab/explain/correct.jpg", width=560))

## 6. Keep the results (optional)

Colab VMs are temporary. Copy the runs and reports to Google Drive, or download them as a zip.

In [ ]:
SAVE_TO_DRIVE = False

!zip -qr food_vision_colab_results.zip runs_colab reports_colab
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp food_vision_colab_results.zip /content/drive/MyDrive/
else:
    from google.colab import files
    files.download("food_vision_colab_results.zip")